# jawiki グラフ分割 PoC — Colab ノートブック

**目的**: jawiki 全記事リンクグラフをコミュニティ分割し、「内部結合が強く外部結合が弱い単位(=銀河)」が得られるかを定量評価する。

構成(§16.3 の骨子に対応):
1. 環境セットアップ
2. `wu` パッケージの書き込み
3. データ取得(Wikimedia ダンプ 4 ファイル, 計 ~1.13GB)
4. ページ正規化(page/redirect/linktarget 解析)
5. リンクグラフ構築(pagelinks ストリーム → 解決済みエッジ)
6. 基本統計(全グラフ次数・ハブ)
7. サブセット抽出(BFS / ID窓)
8. Leiden + コミュニティ指標
9. ヒストグラム表示
10. コミュニティ間エッジ上位表示
11. 階層化テスト(L2)
12. 結論まとめ

注意:
- Colab 無料枠 RAM ~13GB では 100k ノード級サブセットまで安定。全グラフ(142M エッジ, ~10GB)は 32GB ローカル推奨。
- ダウンロードに ~10 分、edges 構築に ~5-15 分(接続速度による)。
- セッションが切れたら「ランタイムを再起動してすべて実行」で再開(ダウンロードは resume 対応)。


In [ ]:
# 1. 環境セットアップ
!pip install -q igraph leidenalg pyarrow
import sys, os
print(sys.version)


In [ ]:
# 2. wu パッケージ用ディレクトリ
import os
os.makedirs("wu", exist_ok=True)
os.makedirs("data", exist_ok=True)
os.makedirs("results", exist_ok=True)


In [ ]:
%%writefile wu/dumpio.py
"""Download + streaming decompression helpers for Wikimedia SQL dumps."""
from __future__ import annotations

import gzip
import hashlib
import json
import os
import threading
import time
import urllib.request

DEFAULT_UA = "WikiGraphResearch/0.1 (https://example.org; research@example.org) python-urllib"
DUMP_BASE = "https://dumps.wikimedia.org/jawiki/latest/"

FILES = {
    "page": "jawiki-latest-page.sql.gz",
    "redirect": "jawiki-latest-redirect.sql.gz",
    "linktarget": "jawiki-latest-linktarget.sql.gz",
    "pagelinks": "jawiki-latest-pagelinks.sql.gz",
    "categorylinks": "jawiki-latest-categorylinks.sql.gz",
}


def download_file(url: str, dest: str, ua: str = DEFAULT_UA, log=print) -> str:
    """Resumable single-file download."""
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    have = os.path.getsize(dest) if os.path.exists(dest) else 0
    headers = {"User-Agent": ua}
    if have:
        headers["Range"] = f"bytes={have}-"
    req = urllib.request.Request(url, headers=headers)
    t0 = time.time()
    with urllib.request.urlopen(req, timeout=120) as r:
        if r.status == 416:  # already complete
            log(f"[skip] {os.path.basename(dest)} already complete")
            return dest
        mode = "ab" if have and r.status == 206 else "wb"
        if mode == "wb":
            have = 0
        total = r.headers.get("Content-Length")
        total = int(total) + have if total else None
        n = have
        last_log = t0
        with open(dest, "ab" if mode == "ab" else "wb") as f:
            while True:
                chunk = r.read(1 << 20)
                if not chunk:
                    break
                f.write(chunk)
                n += len(chunk)
                now = time.time()
                if now - last_log > 20:
                    speed = (n - have) / (now - t0) / 1e6
                    pct = f"{n / total * 100:.1f}%" if total else "?"
                    log(f"  {os.path.basename(dest)}: {n/1e6:.0f}MB {pct} {speed:.1f}MB/s")
                    last_log = now
    log(f"[done] {os.path.basename(dest)} {n/1e6:.1f}MB in {time.time()-t0:.0f}s")
    return dest


def download_all(names, dump_dir: str, workers: int = 3, log=print):
    """Download several dump files in parallel threads (per-connection speed is limited)."""
    os.makedirs(dump_dir, exist_ok=True)
    errs = []

    def _one(key):
        try:
            download_file(DUMP_BASE + FILES[key], os.path.join(dump_dir, FILES[key]), log=log)
        except Exception as e:  # retry once from scratch on failure
            log(f"[retry] {key}: {e}")
            try:
                download_file(DUMP_BASE + FILES[key], os.path.join(dump_dir, FILES[key]), log=log)
            except Exception as e2:
                errs.append((key, e2))

    keys = list(names)
    threads = []
    sem = threading.Semaphore(workers)

    def _wrap(k):
        with sem:
            _one(k)

    for k in keys:
        t = threading.Thread(target=_wrap, args=(k,), daemon=True)
        t.start()
        threads.append(t)
    for t in threads:
        t.join()
    if errs:
        raise RuntimeError(f"download failures: {errs}")


def open_gz_counting(path: str):
    """Yield (decompressed_bytes_read_so_far_callable, fileobj)."""
    f = gzip.open(path, "rb")
    return f


def stream_decompressed(path: str, chunk_bytes: int = 1 << 26, skip_to: int = 0):
    """Yield (offset_after_chunk, chunk) of decompressed bytes.

    If skip_to > 0, fast-forwards (decompress & discard) to that decompressed
    offset before yielding. Used for checkpointed resume.
    """
    off = 0
    with gzip.open(path, "rb") as f:
        if skip_to:
            while off < skip_to:
                want = min(chunk_bytes, skip_to - off)
                data = f.read(want)
                if not data:
                    raise RuntimeError(f"cannot skip to {skip_to}; file ends at {off}")
                off += len(data)
        carry = b""
        while True:
            data = f.read(chunk_bytes)
            if not data:
                if carry:
                    off += len(carry)
                    yield off, carry
                break
            out = carry + data
            off += len(out)
            carry = b""
            yield off, out


def sha256_of(path: str, limit: int | None = None) -> str:
    h = hashlib.sha256()
    n = 0
    with open(path, "rb") as f:
        while True:
            b = f.read(1 << 22)
            if not b:
                break
            h.update(b)
            n += len(b)
            if limit and n >= limit:
                break
    return h.hexdigest()[:16]


def read_json(path, default=None):
    if os.path.exists(path):
        with open(path) as f:
            return json.load(f)
    return default


def write_json(path, obj):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2, default=str)


In [ ]:
%%writefile wu/sqlparse.py
"""Streaming parsers for Wikimedia SQL dumps -> compact numpy/parquet artifacts.

All string titles are hashed with blake2b(digest_size=8) so that joins can be
done with sorted-array searchsorted instead of giant python dicts (low RAM).
"""
from __future__ import annotations

import os
import re
import time

import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq

from .dumpio import read_json, stream_decompressed, write_json
from hashlib import blake2b

# ---------------------------------------------------------------- hashing ---

_ESCAPES = {
    ord("\\"): ord("\\"),  # placeholder, handled by _unescape
}
_UNESC_MAP = {
    "0": b"\0", "b": b"\b", "n": b"\n", "r": b"\r", "t": b"\t",
    "Z": b"\x1a", "\\": b"\\", "'": b"'", '"': b'"',
}
_UNESC_RE = re.compile(rb"\\(.)", re.S)


def unescape(s: bytes) -> bytes:
    if b"\\" not in s:
        return s
    return _UNESC_RE.sub(lambda m: _UNESC_MAP.get(m.group(1).decode("latin1"), m.group(1)), s)


def h64(title: bytes) -> int:
    """Deterministic 64-bit hash of a (canonical, underscore-separated) title."""
    v = int.from_bytes(blake2b(title, digest_size=8).digest(), "big")
    return v if v != 0 else 1  # 0 is the 'missing' sentinel


# --------------------------------------------------------------- growlist ---

class GrowArr:
    """Append-only int array with geometric growth (avoids python-list memory)."""

    def __init__(self, dtype=np.int64, cap=1 << 16):
        self.dtype = np.dtype(dtype)
        self.buf = np.empty(cap, dtype=self.dtype)
        self.n = 0

    def extend(self, arr):
        k = len(arr)
        if self.n + k > len(self.buf):
            newcap = max(len(self.buf) * 2, self.n + k)
            nb = np.empty(newcap, dtype=self.dtype)
            nb[: self.n] = self.buf[: self.n]
            self.buf = nb
        self.buf[self.n : self.n + k] = arr
        self.n += k

    def append(self, v):
        self.extend(np.array([v], dtype=self.dtype))

    def result(self):
        return self.buf[: self.n].copy()


# ------------------------------------------------------- page.sql.gz parse --

# (page_id, page_namespace, page_title, page_is_redirect, ...)
PAGE_RE = re.compile(rb"\((\d+),(-?\d+),'((?:[^'\\]|\\.)*)',(\d+)")


def build_page_artifacts(page_gz: str, parsed_dir: str, meta_path: str, chunk_bytes=1 << 24):
    """Parse page.sql.gz ->
      articles.parquet        : page_id, title(str)  (ns0, non-redirect), sorted by page_id
      ns0_all_hashes.npz      : sorted hashes + page_ids of ALL ns0 pages (articles+redirects)
      article_ids.npy         : sorted page_ids of ns0 non-redirect articles (int64)
    """
    os.makedirs(parsed_dir, exist_ok=True)
    t0 = time.time()
    all_ids = GrowArr(np.int64)
    all_hashes = GrowArr(np.uint64)
    art_ids = GrowArr(np.int64)
    art_hashes = GrowArr(np.uint64)

    pq_path = os.path.join(parsed_dir, "articles.parquet")
    writer = None
    pq_ids, pq_titles = [], []
    counts = {"rows": 0, "ns0": 0, "ns0_redirect": 0, "ns0_article": 0}

    def flush_pq():
        nonlocal writer, pq_ids, pq_titles
        if not pq_ids:
            return
        tbl = pa.table({
            "page_id": pa.array(np.array(pq_ids, dtype=np.int64), type=pa.int64()),
            "title": pa.array(pq_titles, type=pa.string()),
        })
        if writer is None:
            writer = pq.ParquetWriter(pq_path, tbl.schema, compression="zstd")
        writer.write_table(tbl)
        pq_ids, pq_titles = [], []

    for off, chunk in stream_decompressed(page_gz, chunk_bytes):
        c_all_ids, c_all_h, c_art_ids, c_art_h = [], [], [], []
        for m in PAGE_RE.finditer(chunk):
            counts["rows"] += 1
            ns = int(m.group(2))
            if ns != 0:
                continue
            counts["ns0"] += 1
            pid = int(m.group(1))
            title = unescape(m.group(3))
            hv = h64(title)
            c_all_ids.append(pid)
            c_all_h.append(hv)
            is_redir = m.group(4) != b"0"
            if is_redir:
                counts["ns0_redirect"] += 1
            else:
                counts["ns0_article"] += 1
                c_art_ids.append(pid)
                c_art_h.append(hv)
                pq_ids.append(pid)
                pq_titles.append(title.decode("utf-8", "replace"))
        if c_all_ids:
            all_ids.extend(np.fromiter(c_all_ids, dtype=np.int64, count=len(c_all_ids)))
            all_hashes.extend(np.fromiter(c_all_h, dtype=np.uint64, count=len(c_all_h)))
            art_ids.extend(np.fromiter(c_art_ids, dtype=np.int64, count=len(c_art_ids)))
            art_hashes.extend(np.fromiter(c_art_h, dtype=np.uint64, count=len(c_art_h)))
        flush_pq()
        print(f"  page: {counts['rows']:,} rows, ns0={counts['ns0']:,} "
              f"articles={counts['ns0_article']:,} ({time.time()-t0:.0f}s)", flush=True)
    if writer is not None:
        writer.close()

    # sort all-ns0 table by hash for searchsorted joins
    ah = all_hashes.result()
    ai = all_ids.result()
    order = np.argsort(ah, kind="stable")
    ah, ai = ah[order], ai[order]
    np.savez(os.path.join(parsed_dir, "ns0_all_hashes.npz"), hashes=ah, page_ids=ai)

    # articles sorted by page_id (dump is PK order, but sort to be safe) + parallel hash array
    aids = art_ids.result()
    ahas = art_hashes.result()
    o2 = np.argsort(aids, kind="stable")
    np.save(os.path.join(parsed_dir, "article_ids.npy"), aids[o2])
    np.save(os.path.join(parsed_dir, "article_hashes_sorted_by_id.npy"), ahas[o2])

    meta = read_json(meta_path, {}) or {}
    meta["page"] = {**counts, "secs": round(time.time() - t0, 1),
                    "max_page_id": int(aids.max()) if len(aids) else 0}
    write_json(meta_path, meta)
    print(f"[page] {counts} in {time.time()-t0:.0f}s")
    return counts


# ------------------------------------------------- linktarget.sql.gz parse --

LT_RE = re.compile(rb"\((\d+),(-?\d+),'((?:[^'\\]|\\.)*)'")


def build_linktarget_artifacts(lt_gz: str, parsed_dir: str, meta_path: str, chunk_bytes=1 << 24):
    """Parse linktarget.sql.gz -> lt_hash_ns0.npy : dense int64 array indexed by lt_id
    holding blake2b64(lt_title) for ns0 targets, 0 for missing/non-ns0."""
    t0 = time.time()
    ids = GrowArr(np.int64)
    hashes = GrowArr(np.uint64)
    counts = {"rows": 0, "ns0": 0}
    max_id = 0
    for off, chunk in stream_decompressed(lt_gz, chunk_bytes):
        c_ids, c_h = [], []
        for m in LT_RE.finditer(chunk):
            counts["rows"] += 1
            if int(m.group(2)) != 0:
                continue
            counts["ns0"] += 1
            lt_id = int(m.group(1))
            c_ids.append(lt_id)
            c_h.append(h64(unescape(m.group(3))))
            if lt_id > max_id:
                max_id = lt_id
        if c_ids:
            ids.extend(np.fromiter(c_ids, dtype=np.int64, count=len(c_ids)))
            hashes.extend(np.fromiter(c_h, dtype=np.uint64, count=len(c_h)))
            print(f"  linktarget: {counts['rows']:,} rows ns0={counts['ns0']:,} ({time.time()-t0:.0f}s)", flush=True)
    dense = np.zeros(max_id + 2, dtype=np.uint64)
    dense[ids.result()] = hashes.result()
    np.save(os.path.join(parsed_dir, "lt_hash_ns0.npy"), dense)
    meta = read_json(meta_path, {}) or {}
    meta["linktarget"] = {**counts, "max_lt_id": int(max_id), "secs": round(time.time() - t0, 1)}
    write_json(meta_path, meta)
    print(f"[linktarget] {counts} max_lt_id={max_id:,} in {time.time()-t0:.0f}s")
    return counts


# ---------------------------------------------------- redirect.sql.gz parse --

RD_RE = re.compile(rb"\((\d+),(-?\d+),'((?:[^'\\]|\\.)*)','((?:[^'\\]|\\.)*)'")


def build_redirect_artifacts(rd_gz: str, parsed_dir: str, meta_path: str, chunk_bytes=1 << 24):
    """Parse redirect.sql.gz -> rd_map.npz {rd_from(sorted int64), rd_final(int64 page_id, 0=dangling)}

    Targets are resolved against the ns0 hash table; chains are chased up to 10 hops.
    Final targets that are not articles (missing/redirect-loop) become 0.
    """
    z = np.load(os.path.join(parsed_dir, "ns0_all_hashes.npz"))
    all_h, all_pid = z["hashes"], z["page_ids"]
    article_ids = np.load(os.path.join(parsed_dir, "article_ids.npy"))

    t0 = time.time()
    froms = GrowArr(np.int64)
    tos = GrowArr(np.int64)
    counts = {"rows": 0, "ns0_local": 0, "target_found": 0, "dangling": 0}
    for off, chunk in stream_decompressed(rd_gz, chunk_bytes):
        f_list, t_list = [], []
        for m in RD_RE.finditer(chunk):
            counts["rows"] += 1
            if int(m.group(2)) != 0:
                continue
            if m.group(4) not in (b"",):  # rd_interwiki non-empty -> external redirect
                continue
            counts["ns0_local"] += 1
            hv = np.uint64(h64(unescape(m.group(3))))
            pos = np.searchsorted(all_h, hv)
            if pos < len(all_h) and all_h[pos] == hv:
                t_list.append(int(all_pid[pos]))
                counts["target_found"] += 1
            else:
                t_list.append(0)
                counts["dangling"] += 1
            f_list.append(int(m.group(1)))
        if f_list:
            froms.extend(np.array(f_list, dtype=np.int64))
            tos.extend(np.array(t_list, dtype=np.int64))
    rf = froms.result()
    rt = tos.result()
    o = np.argsort(rf, kind="stable")
    rf, rt = rf[o], rt[o]

    # chase chains (vectorized, max 10 hops)
    cur = rt.copy()
    for _ in range(10):
        pos = np.searchsorted(rf, cur)
        pos_c = np.clip(pos, 0, len(rf) - 1) if len(rf) else pos
        if len(rf) == 0:
            break
        hit = (cur != 0) & (rf[pos_c] == cur)
        if not hit.any():
            break
        cur[hit] = rt[pos_c[hit]]
    # final must be an existing article
    pos = np.searchsorted(article_ids, cur)
    pos_c = np.clip(pos, 0, len(article_ids) - 1)
    ok = (cur != 0) & (article_ids[pos_c] == cur)
    final = np.where(ok, cur, 0)
    counts["final_resolved"] = int(ok.sum())
    np.savez(os.path.join(parsed_dir, "rd_map.npz"), rd_from=rf, rd_final=final)
    meta = read_json(meta_path, {}) or {}
    meta["redirect"] = {**counts, "secs": round(time.time() - t0, 1)}
    write_json(meta_path, meta)
    print(f"[redirect] {counts} in {time.time()-t0:.0f}s")
    return counts


In [ ]:
%%writefile wu/buildedges.py
"""Stream pagelinks.sql.gz -> resolved ns0 article->article edge list (int32 binary).

Schema (2026 dumps): pagelinks(pl_from, pl_from_namespace, pl_target_id)
  pl_target_id -> linktarget(lt_id, lt_namespace, lt_title)

Resolution pipeline (fully vectorized per chunk):
  1. keep rows with pl_from_namespace == 0
  2. drop rows whose pl_from is a redirect page (redirect pages are not nodes)
  3. lt_id -> blake2b64(title) via dense array (0 = non-ns0 / missing)
  4. hash -> page_id via searchsorted on sorted ns0-all hash table (miss = red link)
  5. redirect targets resolved via rd_map (dangling -> dropped)
  6. drop self loops; require src to be a non-redirect article
  7. append (src, dst) int32 pairs to edges_ns0.bin

Checkpointed: (decompressed offset, rows seen) persisted so an interrupted run
can resume by fast-forwarding the gzip stream.
"""
from __future__ import annotations

import itertools
import os
import re
import time

import numpy as np

from .dumpio import read_json, stream_decompressed, write_json

PL_RE = re.compile(rb"\((\d+),(-?\d+),(\d+)\)")


def build_edges(pagelinks_gz: str, parsed_dir: str, out_bin: str, ckpt_path: str,
                chunk_bytes: int = 1 << 24, log_every: float = 30.0):
    t0 = time.time()
    z = np.load(os.path.join(parsed_dir, "ns0_all_hashes.npz"))
    all_h, all_pid = z["hashes"], z["page_ids"]
    article_ids = np.load(os.path.join(parsed_dir, "article_ids.npy"))
    lt_hash = np.load(os.path.join(parsed_dir, "lt_hash_ns0.npy"), mmap_mode="r")
    rd = np.load(os.path.join(parsed_dir, "rd_map.npz"))
    rd_from, rd_final = rd["rd_from"], rd["rd_final"]

    ck = read_json(ckpt_path, None)
    counters = {
        "rows_total": 0, "from_non_ns0": 0, "src_is_redirect": 0, "src_not_article": 0,
        "target_non_ns0": 0, "target_redlink": 0, "target_dangling_redirect": 0,
        "self_loops": 0, "kept": 0,
    }
    skip_to = 0
    if ck and ck.get("state") != "done":
        skip_to = ck["decomp_offset"]
        counters.update(ck["counters"])
        print(f"[resume] from decomp_offset={skip_to/1e9:.2f}GB kept={counters['kept']:,}")
    elif ck and ck.get("state") == "done":
        print("[skip] edges already built")
        counters.update(ck.get("counters", {}))
        return counters

    os.makedirs(os.path.dirname(out_bin), exist_ok=True)
    fmode = "r+b" if (skip_to and os.path.exists(out_bin)) else "wb"
    fout = open(out_bin, fmode)
    if skip_to:
        fout.seek(counters["kept"] * 8)
        fout.truncate()

    last_log = t0
    n_lt = len(lt_hash)
    carry = b""

    for off, chunk in stream_decompressed(pagelinks_gz, chunk_bytes, skip_to=skip_to):
        # cut at last complete tuple; carry the remainder to the next chunk
        chunk = carry + chunk
        cut = chunk.rfind(b")")
        if cut < 0:
            carry = chunk
            continue
        carry = chunk[cut + 1 :]
        chunk = chunk[: cut + 1]
        matches = PL_RE.findall(chunk)
        if matches:
            flat = itertools.chain.from_iterable(matches)
            arr = np.fromiter(map(int, flat), dtype=np.int64, count=3 * len(matches)).reshape(-1, 3)
            del matches, flat
            n = len(arr)
            counters["rows_total"] += n

            m = arr[:, 1] == 0
            counters["from_non_ns0"] += int(n - m.sum())
            arr = arr[m]
            src = arr[:, 0]
            lt = arr[:, 2]

            # (2) src must not be a redirect page
            if len(rd_from):
                p = np.searchsorted(rd_from, src)
                pc = np.clip(p, 0, len(rd_from) - 1)
                is_redir = rd_from[pc] == src
            else:
                is_redir = np.zeros(len(src), bool)
            counters["src_is_redirect"] += int(is_redir.sum())
            keep = ~is_redir
            src, lt = src[keep], lt[keep]

            # (6a) src must be an existing article
            p = np.searchsorted(article_ids, src)
            pc = np.clip(p, 0, len(article_ids) - 1)
            ok_src = article_ids[pc] == src
            counters["src_not_article"] += int((~ok_src).sum())
            src, lt = src[ok_src], lt[ok_src]

            # (3) lt_id -> title hash
            valid_lt = (lt >= 0) & (lt < n_lt)
            counters["target_non_ns0"] += int((~valid_lt).sum())
            src, lt = src[valid_lt], lt[valid_lt]
            hv = np.asarray(lt_hash)[lt] if len(lt) else lt
            nz = hv != 0
            counters["target_non_ns0"] += int((~nz).sum())
            src, hv = src[nz], hv[nz]

            # (4) hash -> page_id
            if len(hv):
                p = np.searchsorted(all_h, hv)
                pc = np.clip(p, 0, len(all_h) - 1)
                found = all_h[pc] == hv
                counters["target_redlink"] += int((~found).sum())
                src, dst = src[found], all_pid[pc[found]]

                # (5) redirect resolution
                if len(rd_from) and len(dst):
                    p = np.searchsorted(rd_from, dst)
                    pc = np.clip(p, 0, len(rd_from) - 1)
                    hit = rd_from[pc] == dst
                    if hit.any():
                        fin = rd_final[pc]
                        dst2 = np.where(hit, fin, dst)
                        bad = hit & (fin == 0)
                        counters["target_dangling_redirect"] += int(bad.sum())
                        good = ~bad
                        src, dst = src[good], dst2[good]

                # (6b) self loops
                sl = src == dst
                counters["self_loops"] += int(sl.sum())
                src, dst = src[~sl], dst[~sl]

                if len(src):
                    out = np.empty(len(src) * 2, dtype=np.int32)
                    out[0::2] = src.astype(np.int32)
                    out[1::2] = dst.astype(np.int32)
                    out.tofile(fout)
                    counters["kept"] += len(src)
            # NOTE: `tail` (incomplete tuple remainder) is dropped here because
            # stream_decompressed re-reads whole chunks; to stay exact we keep a
            # module-level carry instead:
        del chunk
        now = time.time()
        if now - last_log > log_every:
            gb = off / 1e9
            speed = (off - skip_to) / max(now - t0, 1e-9) / 1e6
            print(f"  edges: {gb:.2f}GB decomp, rows={counters['rows_total']:,}, "
                  f"kept={counters['kept']:,}, {speed:.0f}MB/s, {now-t0:.0f}s", flush=True)
            fout.flush()
            write_json(ckpt_path, {"decomp_offset": off, "counters": counters, "state": "running"})
            last_log = now

    fout.flush()
    fout.close()
    write_json(ckpt_path, {"decomp_offset": off, "counters": counters, "state": "done",
                           "secs": round(time.time() - t0, 1),
                           "out_bytes": os.path.getsize(out_bin)})
    print(f"[edges] done: {counters}")
    return counters


In [ ]:
%%writefile wu/stats.py
"""Full-graph statistics computed with memmap + chunked passes (low RAM).

Layout policy: reads graph/edges_ns0.bin + parsed/article_ids.npy,
writes graph/indeg.npy, graph/outdeg.npy, graph/full_stats.json.
"""
from __future__ import annotations

import os
import re
import time

import numpy as np

from .dumpio import write_json
from .paths import Dirs

HUB_PATTERNS = {
    "disambig": re.compile(r"\(曖昧さ回避\)|曖昧さ回避$"),
    "list": re.compile(r"一覧$|一覧_|^リスト|の一覧"),
    "year": re.compile(r"^\d{1,4}年$|^\d{1,4}年の|^\d{4}年度"),
    "date": re.compile(r"^\d{1,2}月\d{1,2}日$"),
    "jp_geo": re.compile(r"^日本の|^日本における"),
    "index": re.compile(r"^索引|^Wikipedia:|^Category:"),
}


def hub_flags(title: str) -> list:
    return [k for k, p in HUB_PATTERNS.items() if p.search(title)]


def load_edges_mmap(path: str) -> np.ndarray:
    n = os.path.getsize(path) // 8
    return np.memmap(path, dtype=np.int32, mode="r", shape=(n, 2))


def load_titles(parsed_dir: str):
    import pyarrow.parquet as pq
    tbl = pq.read_table(os.path.join(parsed_dir, "articles.parquet"), columns=["page_id", "title"])
    return tbl.column("page_id").to_numpy(), tbl.column("title")


def full_stats(dirs: Dirs, out_json: str | None = None, titles_needed: int = 200):
    t0 = time.time()
    out_json = out_json or dirs.full_stats
    E = load_edges_mmap(dirs.edges_bin)
    article_ids = np.load(dirs.article_ids)  # sorted page_ids
    n_nodes = len(article_ids)

    # remap page_id -> compact idx (all edge endpoints are articles by construction)
    outdeg = np.zeros(n_nodes, dtype=np.int64)
    indeg = np.zeros(n_nodes, dtype=np.int64)
    CH = 5_000_000
    n_edges = len(E)
    bad = 0
    for i in range(0, n_edges, CH):
        blk = np.asarray(E[i : i + CH]).astype(np.int64)
        s = np.searchsorted(article_ids, blk[:, 0])
        d = np.searchsorted(article_ids, blk[:, 1])
        sc = np.clip(s, 0, n_nodes - 1)
        dc = np.clip(d, 0, n_nodes - 1)
        ok = (article_ids[sc] == blk[:, 0]) & (article_ids[dc] == blk[:, 1])
        bad += int((~ok).sum())
        outdeg += np.bincount(sc[ok], minlength=n_nodes)
        indeg += np.bincount(dc[ok], minlength=n_nodes)

    deg = outdeg + indeg
    dirs.ensure(dirs.graph)
    np.save(dirs.indeg, indeg.astype(np.int32))
    np.save(dirs.outdeg, outdeg.astype(np.int32))
    top = np.argsort(deg)[::-1][:titles_needed]
    pid_col, title_col = load_titles(dirs.parsed)
    top_pid = article_ids[top]
    pos = np.clip(np.searchsorted(pid_col, top_pid), 0, len(pid_col) - 1)
    top_titles = [title_col[p].as_py() for p in pos]

    top_rows = []
    for rank, (p, t, i) in enumerate(zip(top_pid.tolist(), top_titles, top.tolist())):
        top_rows.append({
            "rank": rank + 1, "page_id": p, "title": t,
            "in": int(indeg[i]), "out": int(outdeg[i]), "total": int(deg[i]),
            "flags": hub_flags(t),
        })

    pct = [50, 75, 90, 95, 99, 99.9]
    stats = {
        "n_articles": n_nodes,
        "n_edges_directed_resolved": int(n_edges),
        "endpoint_remap_failures": bad,
        "avg_out_degree": float(n_edges / n_nodes),
        "degree_percentiles": {f"p{p}": float(np.percentile(deg, p)) for p in pct},
        "max_degree": int(deg.max()),
        "n_isolated_articles": int((deg == 0).sum()),
        "edge_share_top0.1pct_nodes": float(np.sort(deg)[::-1][: max(1, n_nodes // 1000)].sum() / max(1, deg.sum())),
        "edge_share_top1pct_nodes": float(np.sort(deg)[::-1][: max(1, n_nodes // 100)].sum() / max(1, deg.sum())),
        "secs": round(time.time() - t0, 1),
    }
    write_json(out_json, {"stats": stats, "top_hubs": top_rows[:60]})
    print(f"[stats] edges={n_edges:,} nodes={n_nodes:,} avg_deg={stats['avg_out_degree']:.1f} "
          f"max_deg={stats['max_degree']:,} in {stats['secs']}s -> {out_json}")
    return stats, top_rows


In [ ]:
%%writefile wu/subsets.py
"""Subset extraction from the full resolved edge list (chunked scans, low RAM).

Strategies (recorded in subset_meta.json):
  * id_window : contiguous window of page_id order (contrast case; leaks ~95%)
  * bfs       : k-hop out-link BFS from seed articles with
                - per-node out-link sampling cap
                - mega-hub exclusion: discovered nodes with full-graph in-degree
                  > max_in_degree are NOT added to the subset (seeds exempt).
                  Rationale: template/year/infobox hubs act as "intergalactic
                  medium", not galaxy members; including them collapses the
                  subset into one blob and explodes boundary traffic.

Outputs per subset dir:
  nodes.parquet        page_id, idx (compact), title
  edges_internal.npy   int32 (n,2) compact idx pairs (directed, reciprocals kept)
  node_stats.npz       ext_out (directed links to outside), ext_in, boundary_any
  subset_meta.json     counts, params, provenance

Layout policy: subsets live under <base>/graph/subsets/<name>/.
"""
from __future__ import annotations

import os
import time

import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq

from .dumpio import write_json
from .stats import load_edges_mmap


def _titles_for(parsed_dir: str, pids: np.ndarray):
    tbl = pq.read_table(os.path.join(parsed_dir, "articles.parquet"), columns=["page_id", "title"])
    pid_col = tbl.column("page_id").to_numpy()
    title_col = tbl.column("title")
    pos = np.clip(np.searchsorted(pid_col, pids), 0, len(pid_col) - 1)
    return [title_col[p].as_py() for p in pos]


def select_id_window(parsed_dir: str, k: int, mode: str = "late"):
    ids = np.load(os.path.join(parsed_dir, "article_ids.npy"))  # sorted page_ids
    if mode == "late":
        return ids[-k:].copy()
    elif mode == "early":
        return ids[:k].copy()
    else:
        m = len(ids) // 2
        return ids[m - k // 2 : m - k // 2 + k].copy()


def title_to_pid(parsed_dir: str, title: str) -> int | None:
    from .sqlparse import h64

    z = np.load(os.path.join(parsed_dir, "ns0_all_hashes.npz"))
    hv = np.uint64(h64(title.replace(" ", "_").encode("utf-8")))
    pos = np.searchsorted(z["hashes"], hv)
    if pos < len(z["hashes"]) and z["hashes"][pos] == hv:
        return int(z["page_ids"][pos])
    return None


def _pid_to_idx(parsed_dir: str, pids: np.ndarray) -> np.ndarray:
    article_ids = np.load(os.path.join(parsed_dir, "article_ids.npy"))
    return np.clip(np.searchsorted(article_ids, pids), 0, len(article_ids) - 1).astype(np.int64)


def bfs_nodes(edges_bin: str, seed_pids, max_nodes: int, cap_per_node: int = 40,
              max_hops: int = 4, rng_seed: int = 42, parsed_dir: str | None = None,
              graph_dir: str | None = None, max_in_degree: int | None = 10000):
    """Out-link BFS with per-node sampling cap and mega-hub exclusion."""
    rng = np.random.default_rng(rng_seed)
    E = load_edges_mmap(edges_bin)

    indeg = None
    article_ids = None
    if parsed_dir and graph_dir and max_in_degree:
        p = os.path.join(graph_dir, "indeg.npy")
        if os.path.exists(p):
            indeg = np.load(p)
            article_ids = np.load(os.path.join(parsed_dir, "article_ids.npy"))

    visited = set(int(s) for s in seed_pids)
    frontier = np.array(sorted(visited), dtype=np.int64)
    info = {"hops": [], "excluded_hubs": 0, "max_in_degree": max_in_degree}
    for hop in range(max_hops):
        if len(frontier) == 0 or len(visited) >= max_nodes:
            break
        fs = np.sort(frontier)
        pairs_src, pairs_dst = [], []
        CH = 4_000_000
        for i in range(0, len(E), CH):
            blk = np.asarray(E[i : i + CH]).astype(np.int64)
            pos = np.searchsorted(fs, blk[:, 0])
            pc = np.clip(pos, 0, len(fs) - 1)
            m = fs[pc] == blk[:, 0]
            if m.any():
                pairs_src.append(blk[m, 0])
                pairs_dst.append(blk[m, 1])
        if not pairs_src:
            break
        src = np.concatenate(pairs_src)
        dst = np.concatenate(pairs_dst)
        del pairs_src, pairs_dst
        # cap per source: shuffle, then keep first `cap` per src group
        order = rng.permutation(len(src))
        src, dst = src[order], dst[order]
        uniq, inv = np.unique(src, return_inverse=True)
        o2 = np.argsort(inv, kind="stable")
        inv_sorted = inv[o2]
        starts = np.searchsorted(inv_sorted, np.arange(len(uniq)), side="left")
        within = np.arange(len(src)) - starts[inv_sorted]
        rank = np.empty(len(src), dtype=np.int64)
        rank[o2] = within
        m = rank < cap_per_node
        dst = dst[m]
        # mega-hub exclusion (applies to discovered nodes, not seeds)
        if indeg is not None and len(dst):
            pos = np.clip(np.searchsorted(article_ids, dst), 0, len(article_ids) - 1)
            deg = indeg[pos]
            ok = deg <= max_in_degree
            info["excluded_hubs"] += int((~ok).sum())
            dst = dst[ok]
        # new nodes not yet visited
        vs = np.fromiter(visited, dtype=np.int64, count=len(visited))
        vs.sort()
        pos = np.searchsorted(vs, dst)
        pc = np.clip(pos, 0, len(vs) - 1)
        not_in = ~((pos < len(vs)) & (vs[pc] == dst))
        new = np.unique(dst[not_in])
        room = max_nodes - len(visited)
        if len(new) > room:
            new = np.sort(new[rng.permutation(len(new))[:room]])
        visited.update(new.tolist())
        info["hops"].append({"hop": hop + 1, "frontier": int(len(frontier)),
                             "new_nodes": int(len(new)), "total": len(visited)})
        frontier = new
    return np.array(sorted(visited), dtype=np.int64), info


def extract_subset(edges_bin: str, node_pids: np.ndarray, out_dir: str, meta: dict,
                   parsed_dir: str | None = None):
    """One full scan: collect internal edges; count per-node external traffic."""
    t0 = time.time()
    os.makedirs(out_dir, exist_ok=True)
    nodes = np.sort(np.unique(node_pids)).astype(np.int64)
    n = len(nodes)
    E = load_edges_mmap(edges_bin)
    CH = 4_000_000
    ii_s, ii_d = [], []
    ext_out = np.zeros(n, np.int32)
    ext_in = np.zeros(n, np.int32)
    bnd_any = np.zeros(n, bool)
    n_int = n_out = n_in = 0
    for i in range(0, len(E), CH):
        blk = np.asarray(E[i : i + CH]).astype(np.int64)
        s, d = blk[:, 0], blk[:, 1]
        ps = np.searchsorted(nodes, s)
        pds = np.searchsorted(nodes, d)
        ms = (ps < n) & (nodes[np.clip(ps, 0, n - 1)] == s)
        md = (pds < n) & (nodes[np.clip(pds, 0, n - 1)] == d)
        both = ms & md
        if both.any():
            ii_s.append(ps[both].astype(np.int32))
            ii_d.append(pds[both].astype(np.int32))
            n_int += int(both.sum())
        only_s = ms & ~md
        if only_s.any():
            np.add.at(ext_out, ps[only_s], 1)
            bnd_any[ps[only_s]] = True
            n_out += int(only_s.sum())
        only_d = md & ~ms
        if only_d.any():
            np.add.at(ext_in, pds[only_d], 1)
            bnd_any[pds[only_d]] = True
            n_in += int(only_d.sum())

    internal = np.stack([np.concatenate(ii_s), np.concatenate(ii_d)], axis=1).astype(np.int32) \
        if n_int else np.zeros((0, 2), np.int32)
    np.save(os.path.join(out_dir, "edges_internal.npy"), internal)
    np.savez(os.path.join(out_dir, "node_stats.npz"),
             ext_out=ext_out, ext_in=ext_in, boundary_any=bnd_any)

    titles = _titles_for(parsed_dir, nodes) if parsed_dir else [""] * n
    tbl = pa.table({
        "page_id": pa.array(nodes, type=pa.int64()),
        "idx": pa.array(np.arange(n, dtype=np.int32), type=pa.int32()),
        "title": pa.array(titles, type=pa.string()),
    })
    pq.write_table(tbl, os.path.join(out_dir, "nodes.parquet"), compression="zstd")

    meta = dict(meta)
    meta.update({
        "n_nodes": n,
        "n_edges_internal_directed": n_int,
        "n_edges_outgoing": n_out,
        "n_edges_incoming": n_in,
        "leak_ratio_out": float(n_out / max(1, n_int + n_out)),
        "boundary_any_nodes": int(bnd_any.sum()),
        "boundary_any_ratio": float(bnd_any.sum() / max(1, n)),
        "build_secs": round(time.time() - t0, 1),
        "page_id_min": int(nodes.min()), "page_id_max": int(nodes.max()),
    })
    write_json(os.path.join(out_dir, "subset_meta.json"), meta)
    print(f"[subset] {out_dir}: nodes={n:,} internal={n_int:,} "
          f"out={n_out:,} in={n_in:,} leak_out={meta['leak_ratio_out']:.3f} ({meta['build_secs']}s)")
    return meta


In [ ]:
%%writefile wu/analysis.py
"""Community detection + independence metrics + hierarchy test.

Metric definitions (matching the PoC spec):
  N_c        nodes in community c
  E_in_c     undirected unique edges with both ends in c
  E_out_c    undirected unique edges with exactly one end in c (other end in
             another community of the same subset)   [inter-community cut]
  out_ratio_c       = E_out_c / (E_in_c + E_out_c)
  conductance_c     = E_out_c / (2*E_in_c + E_out_c)          (= cut / vol(c))
  avg_degree_c      = 2*E_in_c / N_c
  boundary_nodes_c  = nodes of c having >=1 edge leaving c (inter-community)
  boundary_node_ratio_c = boundary_nodes_c / N_c
  E_ext_c    undirected unique links between nodes of c and nodes OUTSIDE the
             subset (boundary pairs, deduped across directions)  [leakage]
  boundary_any_nodes_c = nodes of c with inter-community OR outside-subset links

All "edge counts" are unique undirected article links unless noted.
"""
from __future__ import annotations

import os
import time

import ctypes
import gc

import igraph as ig
import leidenalg
import numpy as np

try:
    _LIBC = ctypes.CDLL("libc.so.6")
except OSError:  # non-linux
    _LIBC = None


def trim():
    """Return freed heap pages to the OS (critical in low-RAM sandboxes)."""
    gc.collect()
    if _LIBC is not None:
        _LIBC.malloc_trim(0)
import pyarrow as pa
import pyarrow.parquet as pq

from .dumpio import read_json, write_json
from .stats import hub_flags

SEED = 42


# ------------------------------------------------------------------ loading --

def load_subset(subset_dir: str):
    nodes = pq.read_table(os.path.join(subset_dir, "nodes.parquet")).to_pydict()
    n = len(nodes["idx"])
    internal = np.load(os.path.join(subset_dir, "edges_internal.npy"))
    ns = np.load(os.path.join(subset_dir, "node_stats.npz"))
    node_stats = {"ext_out": ns["ext_out"].astype(np.int64),
                  "ext_in": ns["ext_in"].astype(np.int64),
                  "boundary_any": ns["boundary_any"]}
    return n, nodes, internal, node_stats


def undirected_unique(internal: np.ndarray) -> np.ndarray:
    """Dedupe directed pairs into unique undirected int32 edges (low RAM)."""
    if len(internal) == 0:
        return np.zeros((0, 2), np.int32)
    a = internal[:, 0]
    b = internal[:, 1]
    lo = np.minimum(a, b)
    hi = np.maximum(a, b)
    del a, b
    mx = int(max(lo.max(), hi.max())) + 1
    if mx <= (1 << 20):  # n <= ~1M: pack into int64 (shift 20 bits)
        keys = lo.astype(np.int64)
        keys <<= 20
        keys |= hi
        del lo, hi
        uq = np.unique(keys)
        del keys
        out = np.empty((len(uq), 2), np.int32)
        out[:, 0] = (uq >> 20).astype(np.int32)
        out[:, 1] = (uq & ((1 << 20) - 1)).astype(np.int32)
        del uq
        return out
    pairs = np.unique(np.stack([lo, hi], axis=1), axis=0)
    return pairs.astype(np.int32)


# ---------------------------------------------------------------- detection --

def build_igraph_batched(n: int, edges: np.ndarray, weights=None, batch: int = 300_000):
    """Memory-safe igraph construction. NOTE: python-igraph's sparse-matrix and
    generator paths materialize giant python lists; batched add_edges + malloc_trim
    keeps RSS bounded. edges may be a memmap."""
    g = ig.Graph(n=n)
    m = len(edges)
    for i in range(0, m, batch):
        blk = np.asarray(edges[i : i + batch])
        g.add_edges(list(zip(blk[:, 0].tolist(), blk[:, 1].tolist())))
        if (i // batch) % 10 == 0:
            trim()
    if weights is not None:
        g.es["weight"] = np.asarray(weights, dtype=np.float64).tolist()
    trim()
    return g


def run_leiden(n: int, edges: np.ndarray, resolution: float = 1.0, seed: int = SEED,
               weights=None, engine: str = "igraph"):
    """engine='igraph': native C Leiden (low RAM, ~58B/edge total; recommended).
    engine='leidenalg': python leidenalg (copies graph internally, ~2x RAM)."""
    g = build_igraph_batched(n, edges, weights)
    if engine == "igraph":
        import random as _random
        _random.seed(seed)
        cl = g.community_leiden(
            "modularity", weights="weight" if weights is not None else None,
            resolution=resolution, n_iterations=3,
        )
        memb = np.asarray(cl.membership, dtype=np.int32)
        q, mod = float(cl.quality), float(cl.modularity)
        del g, cl
    else:
        part = leidenalg.find_partition(
            g, leidenalg.RBConfigurationVertexPartition,
            weights="weight" if weights is not None else None,
            resolution_parameter=resolution, seed=seed,
        )
        memb = np.asarray(part.membership, dtype=np.int32)
        q, mod = part.quality(), part.modularity
        del g, part
    trim()
    return memb, q, mod


# ------------------------------------------------------------------ metrics --

def community_metrics(n: int, E: np.ndarray, memb: np.ndarray,
                      node_stats: dict, nodes: dict):
    """E: undirected unique edges (compact idx). Returns (per_comm dict arrays, globals)."""
    t0 = time.time()
    C = int(memb.max()) + 1
    u, v = E[:, 0], E[:, 1]
    mu, mv = memb[u], memb[v]
    same = mu == mv

    e_in = np.bincount(mu[same], minlength=C).astype(np.int64)
    e_out = np.zeros(C, np.int64)
    cross_u, cross_v = u[~same], v[~same]
    if len(cross_u):
        e_out += np.bincount(mu[~same], minlength=C)
        e_out += np.bincount(mv[~same], minlength=C)
    n_c = np.bincount(memb, minlength=C).astype(np.int64)

    # boundary nodes (inter-community)
    b_inter = np.zeros(n, bool)
    if len(cross_u):
        b_inter[cross_u] = True
        b_inter[cross_v] = True
    bn_inter = np.bincount(memb[b_inter], minlength=C).astype(np.int64)

    # outside-subset traffic (per-node directed counts) -> per-community
    ext_out_c = np.bincount(memb, weights=node_stats["ext_out"], minlength=C).astype(np.int64)
    ext_in_c = np.bincount(memb, weights=node_stats["ext_in"], minlength=C).astype(np.int64)
    ext_c = ext_out_c + ext_in_c
    b_any = b_inter | node_stats["boundary_any"]
    bn_any = np.bincount(memb[b_any], minlength=C).astype(np.int64)

    with np.errstate(divide="ignore", invalid="ignore"):
        out_ratio = np.where(e_in + e_out > 0, e_out / np.maximum(1, e_in + e_out), np.nan)
        conduct = np.where(2 * e_in + e_out > 0, e_out / np.maximum(1, 2 * e_in + e_out), np.nan)
        avg_deg = 2 * e_in / np.maximum(1, n_c)
        ext_ratio = np.where(e_in + e_out + ext_out_c > 0,
                             ext_out_c / np.maximum(1, e_in + e_out + ext_out_c), np.nan)

    per = {
        "comm": np.arange(C), "N_c": n_c, "E_in_c": e_in, "E_out_c": e_out,
        "E_ext_c": ext_c, "E_ext_out_c": ext_out_c, "E_ext_in_c": ext_in_c,
        "out_ratio_c": out_ratio, "conductance_c": conduct,
        "avg_degree_c": avg_deg, "boundary_nodes_c": bn_inter,
        "boundary_node_ratio_c": bn_inter / np.maximum(1, n_c),
        "boundary_any_nodes_c": bn_any, "ext_ratio_c": ext_ratio,
    }

    total_cross = int((~same).sum())
    glob = {
        "n_communities": C,
        "n_nodes": n,
        "n_edges_internal_undirected": int(len(E)),
        "n_cross_community_edges": total_cross,
        "n_edges_to_outside_subset": int(node_stats["ext_out"].sum()),
        "n_edges_from_outside_subset": int(node_stats["ext_in"].sum()),
        "largest_comm_share": float(n_c.max() / n),
        "top5_comm_share": float(np.sort(n_c)[::-1][:5].sum() / n),
        "cross_edge_fraction": float(total_cross / max(1, len(E))),
        "mean_out_ratio_edge_weighted": float(e_out.sum() / max(1, e_in.sum() + e_out.sum())),
        "size_percentiles": {f"p{p}": float(np.percentile(n_c, p)) for p in (5, 25, 50, 75, 95)},
        "out_ratio_percentiles": {f"p{p}": float(np.nanpercentile(out_ratio, p)) for p in (10, 25, 50, 75, 90)},
        "conductance_percentiles": {f"p{p}": float(np.nanpercentile(conduct, p)) for p in (10, 25, 50, 75, 90)},
        "node_weighted_median_out_ratio": float(_weighted_median(out_ratio, n_c)),
        "secs": round(time.time() - t0, 2),
    }
    return per, glob


def _weighted_median(values: np.ndarray, weights: np.ndarray) -> float:
    m = ~np.isnan(values) & (weights > 0)
    if not m.any():
        return float("nan")
    vals, w = values[m], weights[m].astype(float)
    o = np.argsort(vals)
    vals, w = vals[o], w[o]
    cw = np.cumsum(w)
    return float(vals[np.searchsorted(cw, cw[-1] / 2)])


def top_inter_pairs(E: np.ndarray, memb: np.ndarray, top_k: int = 20):
    C = int(memb.max()) + 1
    u, v = E[:, 0].astype(np.int64), E[:, 1].astype(np.int64)
    mu, mv = memb[u], memb[v]
    cross = mu != mv
    a = np.minimum(mu[cross], mv[cross])
    b = np.maximum(mu[cross], mv[cross])
    keys = a * C + b
    uk, cnt = np.unique(keys, return_counts=True)
    o = np.argsort(cnt)[::-1][:top_k]
    out = []
    for k, c in zip(uk[o], cnt[o]):
        out.append({"comm_a": int(k // C), "comm_b": int(k % C), "edges": int(c)})
    return out


def community_labels(n: int, E: np.ndarray, memb: np.ndarray, nodes: dict, per, top_n=3):
    """Representative titles per community: highest internal-degree members."""
    deg = np.zeros(n, np.int64)
    if len(E):
        deg += np.bincount(E[:, 0], minlength=n)
        deg += np.bincount(E[:, 1], minlength=n)
    C = len(per["comm"])
    labels = {}
    order = np.argsort(-deg)
    seen = {c: 0 for c in range(C)}
    titles = nodes["title"]
    for i in order:
        c = int(memb[i])
        if seen[c] < top_n:
            labels.setdefault(c, []).append(titles[i])
            seen[c] += 1
        if all(v >= top_n for v in seen.values()):
            break
    return {c: labels.get(c, []) for c in range(C)}


def hub_analysis(n: int, E: np.ndarray, memb: np.ndarray, nodes: dict, node_stats: dict, top_k=30):
    deg_in = np.zeros(n, np.int64)
    if len(E):
        deg_in += np.bincount(E[:, 0], minlength=n)
        deg_in += np.bincount(E[:, 1], minlength=n)
    pid = np.asarray(nodes["page_id"], dtype=np.int64)
    deg_ext = node_stats["ext_out"] + node_stats["ext_in"]
    total = deg_in + deg_ext
    o = np.argsort(-total)[:top_k]
    titles = nodes["title"]
    rows = []
    for r, i in enumerate(o):
        rows.append({
            "rank": r + 1, "idx": int(i), "title": titles[i], "page_id": int(pid[i]),
            "deg_internal": int(deg_in[i]), "deg_outside": int(deg_ext[i]),
            "comm": int(memb[i]), "flags": hub_flags(titles[i]),
        })
    # concentration: share of internal edges incident to top-1% degree nodes
    k1 = max(1, n // 100)
    top_nodes = np.argsort(-deg_in)[:k1]
    mask = np.zeros(n, bool)
    mask[top_nodes] = True
    if len(E):
        share = float((mask[E[:, 0]] | mask[E[:, 1]]).sum() / len(E))
    else:
        share = 0.0
    stats = {
        "top1pct_internal_edge_incidence_share": share,
        "degree_percentiles": {f"p{p}": float(np.percentile(deg_in, p)) for p in (50, 75, 90, 99)},
        "max_internal_degree": int(deg_in.max()) if n else 0,
    }
    return rows, stats


# ---------------------------------------------------------------- hierarchy --

def pair_matrix(E: np.ndarray, memb: np.ndarray):
    """Return (a, b, w) arrays of the weighted inter-community graph."""
    C = int(memb.max()) + 1
    u, v = E[:, 0].astype(np.int64), E[:, 1].astype(np.int64)
    mu, mv = memb[u], memb[v]
    cross = mu != mv
    a = np.minimum(mu[cross], mv[cross])
    b = np.maximum(mu[cross], mv[cross])
    keys = a * C + b
    uk, cnt = np.unique(keys, return_counts=True)
    return uk // C, uk % C, cnt.astype(np.float64), C


def hierarchy_level2(memb: np.ndarray, E: np.ndarray, per: dict, resolution: float = 1.0,
                     seed: int = SEED):
    a, b, w, C = pair_matrix(E, memb)
    if C < 2 or len(a) == 0:
        return None
    g = ig.Graph(n=C, edges=list(zip(a.tolist(), b.tolist())), directed=False)
    g.es["weight"] = w.tolist()
    part = leidenalg.find_partition(g, leidenalg.RBConfigurationVertexPartition,
                                    weights="weight", resolution_parameter=resolution, seed=seed)
    cl = np.asarray(part.membership, dtype=np.int32)
    K = int(cl.max()) + 1

    e_in_c = per["E_in_c"]
    ext_c = per["E_ext_c"]
    n_c = per["N_c"]
    # level2 aggregates
    E_in2 = np.bincount(cl, weights=e_in_c, minlength=K)
    E_between2 = np.zeros(K)  # intra-cluster inter-community weights
    E_out2 = np.zeros(K)      # inter-cluster weights
    for x, y, wt in zip(a.tolist(), b.tolist(), w.tolist()):
        cx, cy = cl[x], cl[y]
        if cx == cy:
            E_between2[cx] += wt
        else:
            E_out2[cx] += wt
            E_out2[cy] += wt
    N2 = np.bincount(cl, weights=n_c, minlength=K).astype(np.int64)
    E_ext2 = np.bincount(cl, weights=ext_c, minlength=K)
    ncomms = np.bincount(cl, minlength=K)
    with np.errstate(divide="ignore", invalid="ignore"):
        ein2_total = E_in2 + E_between2
        out_ratio2 = np.where(ein2_total + E_out2 > 0, E_out2 / np.maximum(1, ein2_total + E_out2), np.nan)
        conduct2 = np.where(2 * ein2_total + E_out2 > 0, E_out2 / np.maximum(1, 2 * ein2_total + E_out2), np.nan)

    total_intra_w = float(E_in2.sum() + E_between2.sum())
    glob = {
        "n_clusters": K,
        "resolution": resolution,
        "modularity_community_graph": float(part.modularity),
        "mean_out_ratio2_edge_weighted": float(E_out2.sum() / max(1e-9, total_intra_w + E_out2.sum())),
        "largest_cluster_share": float(N2.max() / max(1, N2.sum())),
        "out_ratio2_percentiles": {f"p{p}": float(np.nanpercentile(out_ratio2, p)) for p in (10, 50, 90)} if K > 2 else {},
    }
    rows = []
    for k in range(K):
        rows.append({
            "cluster": k, "n_comms": int(ncomms[k]), "N_nodes": int(N2[k]),
            "E_in_articles": float(E_in2[k]), "E_between_comms": float(E_between2[k]),
            "E_out2": float(E_out2[k]), "E_ext_outside_subset": float(E_ext2[k]),
            "out_ratio2": float(out_ratio2[k]) if not np.isnan(out_ratio2[k]) else None,
            "conductance2": float(conduct2[k]) if not np.isnan(conduct2[k]) else None,
        })
    return {"clusters": rows, "global": glob, "comm_to_cluster": cl.tolist()}


In [ ]:
%%writefile wu/pipeline.py
"""Analyze one subset end-to-end: Leiden sweep -> pick primary -> metrics ->
hubs -> hierarchy -> plots -> report.md + metrics.json."""
from __future__ import annotations

import datetime
import json
import os
import time

import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq

from . import analysis as an
from .dumpio import read_json, write_json
from .paths import Dirs


def _pick_primary(sweep_results, target_median=(200, 3000), ideal=800):
    """Pick the resolution whose median community size is closest to `ideal`
    (log-scale) inside the band; degenerate results (median<50) are penalized."""
    best, best_d = None, 1e18
    for s in sweep_results:
        med = s["p50"]
        d = abs(np.log(max(med, 1) / ideal))
        if med < 50:
            score = 1000 + d
        elif target_median[0] <= med <= target_median[1]:
            score = d
        else:
            score = 100 + d
        if score < best_d:
            best, best_d = s, score
    return best


def filter_edges_by_hub(E: np.ndarray, nodes: dict, dirs: Dirs,
                        indeg_cap: int | None = None, outdeg_cap: int | None = None):
    """Drop edges whose endpoint is a mega-hub in the FULL graph (layout-graph
    style hub suppression). Returns (E_filtered, info)."""
    if indeg_cap is None and outdeg_cap is None:
        return E, {"dropped": 0}
    article_ids = np.load(dirs.article_ids)
    pid = np.asarray(nodes["page_id"], dtype=np.int64)
    pos = np.clip(np.searchsorted(article_ids, pid), 0, len(article_ids) - 1)
    keep_n = np.ones(len(pid), bool)
    info = {}
    if indeg_cap is not None:
        indeg = np.load(dirs.indeg)[pos]
        info["nodes_indeg_capped"] = int((indeg > indeg_cap).sum())
        keep_n &= indeg <= indeg_cap
    if outdeg_cap is not None:
        outdeg = np.load(dirs.outdeg)[pos]
        info["nodes_outdeg_capped"] = int((outdeg > outdeg_cap).sum())
        keep_n &= outdeg <= outdeg_cap
    m = keep_n[E[:, 0]] & keep_n[E[:, 1]]
    info["edges_dropped"] = int((~m).sum())
    info["edges_dropped_frac"] = float((~m).sum() / max(1, len(E)))
    info["nodes_isolated_by_filter"] = int((~keep_n).sum())
    return E[m], info


def prune_edges_degree_budget(E: np.ndarray, budget: int, seed: int = an.SEED,
                              mode: str = "both"):
    """Degree-budget pruning of the layout graph.

    modes:
      both : keep edge iff random-rank < budget at BOTH endpoints
             (hard cap per node; fragments star-peripheries -> not recommended)
      smart: keep edge iff rank<budget at either endpoint OR either endpoint has
             degree<=budget. Rationale: galaxy structure lives in low-degree
             (stub<->local-hub) edges; only hub<->hub "intergalactic highways"
             are thinned. Preserves connectivity of the periphery.
    """
    rng = np.random.default_rng(seed)
    E = np.asarray(E)
    n_edges = len(E)
    if n_edges == 0:
        return E, {"budget": budget, "mode": mode, "kept": 0}
    n = int(E.max()) + 1
    deg = np.bincount(E[:, 0], minlength=n) + np.bincount(E[:, 1], minlength=n)
    prio = rng.random(n_edges)

    def _ranks(col):
        node = E[:, col]
        order = np.lexsort((prio, node))
        node_sorted = node[order]
        starts = np.searchsorted(node_sorted, np.arange(n), side="left")
        rank = np.empty(n_edges, dtype=np.int64)
        rank[order] = np.arange(n_edges) - starts[node_sorted]
        return node, rank

    if mode == "both":
        keep = np.ones(n_edges, bool)
        for col in (0, 1):
            _, rank = _ranks(col)
            keep &= rank < budget
    elif mode == "smart":
        keep = np.zeros(n_edges, bool)
        for col in (0, 1):
            node, rank = _ranks(col)
            keep |= (rank < budget) | (deg[node] <= budget)
    else:
        raise ValueError(f"unknown mode: {mode}")
    info = {"budget": budget, "mode": mode, "kept": int(keep.sum()),
            "kept_frac": float(keep.mean())}
    return E[keep], info


def analyze_subset(subset_dir: str, out_dir: str, resolutions=(0.5, 1.0, 2.0),
                   seed: int = an.SEED, full_stats_json: str | None = None,
                   dirs: Dirs | None = None, hub_indeg_cap=None, hub_outdeg_cap=None,
                   degree_budget: int | None = None, hub_weight: str | None = None,
                   degree_budget_mode: str = "smart", engine: str = "igraph"):
    t_start = time.time()
    os.makedirs(out_dir, exist_ok=True)
    n, nodes, internal, node_stats = an.load_subset(subset_dir)
    E_raw = an.undirected_unique(internal)
    del internal
    os.makedirs(out_dir, exist_ok=True)
    E_raw_path = os.path.join(out_dir, "E_raw.npy")
    np.save(E_raw_path, E_raw)
    del E_raw
    an.trim()
    E_raw = np.load(E_raw_path, mmap_mode="r")  # file-backed: reclaimable under pressure
    E = E_raw
    subset_meta = read_json(os.path.join(subset_dir, "subset_meta.json"), {}) or {}
    if full_stats_json is None and dirs is not None:
        full_stats_json = dirs.full_stats
    hub_filter_info = None
    prune_info = None
    if degree_budget:
        E, prune_info = prune_edges_degree_budget(E, degree_budget, seed=seed,
                                                  mode=degree_budget_mode)
        subset_meta = dict(subset_meta)
        subset_meta["degree_budget"] = prune_info
        print(f"[degree-budget {degree_budget_mode}] B={degree_budget}: kept {prune_info['kept']:,} "
              f"({prune_info['kept_frac']:.1%}) edges")
    if dirs and (hub_indeg_cap or hub_outdeg_cap):
        E, hub_filter_info = filter_edges_by_hub(E, nodes, dirs, hub_indeg_cap, hub_outdeg_cap)
        subset_meta = dict(subset_meta)
        subset_meta["hub_filter"] = {"indeg_cap": hub_indeg_cap, "outdeg_cap": hub_outdeg_cap,
                                     **hub_filter_info}
        print(f"[hub-filter] indeg<={hub_indeg_cap} outdeg<={hub_outdeg_cap}: "
              f"dropped {hub_filter_info['edges_dropped']:,} edges "
              f"({hub_filter_info['edges_dropped_frac']:.1%}), E={len(E):,}")
    edge_w = None
    if hub_weight and dirs:
        article_ids = np.load(dirs.article_ids)
        pid = np.asarray(nodes["page_id"], dtype=np.int64)
        pos = np.clip(np.searchsorted(article_ids, pid), 0, len(article_ids) - 1)
        deg = (np.load(dirs.indeg)[pos].astype(np.float64)
               + np.load(dirs.outdeg)[pos].astype(np.float64))
        du, dv = deg[E[:, 0]], deg[E[:, 1]]
        if hub_weight == "log":
            edge_w = 1.0 / (np.log(2.0 + du) * np.log(2.0 + dv))
            formula = "1/(log(2+d_u)*log(2+d_v))"
        elif hub_weight.startswith("deg"):
            alpha = float(hub_weight[3:] or 1.0)
            edge_w = 1.0 / np.power(np.maximum(du, 1.0) * np.maximum(dv, 1.0), alpha)
            formula = f"1/(d_u*d_v)^{alpha}"
        else:
            raise ValueError(f"unknown hub_weight mode: {hub_weight}")
        subset_meta = dict(subset_meta)
        subset_meta["hub_weight"] = {"formula": formula, "mode": hub_weight,
                                     "w_p5": float(np.percentile(edge_w, 5)),
                                     "w_p50": float(np.percentile(edge_w, 50)),
                                     "w_p95": float(np.percentile(edge_w, 95)),
                                     "w_max": float(edge_w.max())}
        print(f"[hub-weight] {formula}; w p5/p50/p95/max = {subset_meta['hub_weight']['w_p5']:.2e}/"
              f"{subset_meta['hub_weight']['w_p50']:.2e}/{subset_meta['hub_weight']['w_p95']:.2e}/"
              f"{subset_meta['hub_weight']['w_max']:.2e}")
    print(f"[analyze] {subset_dir}: n={n:,} E_undirected={len(E):,} "
          f"ext_out={int(node_stats['ext_out'].sum()):,}")

    # ---- resolution sweep
    sweep = []
    memb_by_res = {}
    for res in resolutions:
        t0 = time.time()
        memb, quality, modularity = an.run_leiden(n, E, resolution=res, seed=seed,
                                                  weights=edge_w, engine=engine)
        per, glob = an.community_metrics(n, E, memb, node_stats, nodes)
        sizes = per["N_c"][per["N_c"] > 0]
        sweep.append({
            "resolution": res, "n_communities": glob["n_communities"],
            "largest_comm_share": glob["largest_comm_share"],
            "top5_comm_share": glob["top5_comm_share"],
            "cross_edge_fraction": glob["cross_edge_fraction"],
            "mean_out_ratio_edge_weighted": glob["mean_out_ratio_edge_weighted"],
            "p25": float(np.percentile(sizes, 25)), "p50": float(np.percentile(sizes, 50)),
            "p75": float(np.percentile(sizes, 75)),
            "quality": float(quality), "modularity": float(modularity),
            "secs": round(time.time() - t0, 1),
        })
        memb_by_res[res] = memb
        an.trim()
        print(f"  res={res}: C={glob['n_communities']} largest={glob['largest_comm_share']:.3f} "
              f"median_size={sweep[-1]['p50']:.0f} cross_frac={glob['cross_edge_fraction']:.3f} "
              f"({sweep[-1]['secs']}s)")

    primary = _pick_primary(sweep)
    res = primary["resolution"]
    memb = memb_by_res[res]
    print(f"[analyze] primary resolution={res} (median size {primary['p50']:.0f})")

    # ---- full metrics at primary resolution
    per, glob = an.community_metrics(n, E, memb, node_stats, nodes)
    if E is not E_raw and len(E_raw):
        u, v = E_raw[:, 0], E_raw[:, 1]
        same = memb[u] == memb[v]
        glob["raw_eval"] = {
            "n_edges_raw": int(len(E_raw)),
            "cross_edge_fraction_raw": float((~same).sum() / len(E_raw)),
        }
        del u, v, same
        an.trim()
    labels = an.community_labels(n, E, memb, nodes, per, top_n=3)
    top_pairs = an.top_inter_pairs(E, memb, top_k=20)
    hub_rows, hub_stats = an.hub_analysis(n, E, memb, nodes, node_stats, top_k=30)
    hier = an.hierarchy_level2(memb, E, per, resolution=1.0, seed=seed)

    # ---- persist
    pid = np.asarray(nodes["page_id"], dtype=np.int64)
    cl = np.asarray(hier["comm_to_cluster"], dtype=np.int32) if hier else np.full(glob["n_communities"], -1, np.int32)
    tbl = pa.table({
        "idx": pa.array(np.arange(n, dtype=np.int32)),
        "page_id": pa.array(pid),
        "title": pa.array(nodes["title"], type=pa.string()),
        "comm": pa.array(memb),
        "cluster": pa.array(cl[memb]),
    })
    pq.write_table(tbl, os.path.join(out_dir, "communities.parquet"), compression="zstd")

    per_json = {}
    for k, v in per.items():
        arr = np.asarray(v)
        per_json[k] = [None if (isinstance(x, float) and np.isnan(x)) else (int(x) if arr.dtype.kind == "i" else float(x)) for x in arr.tolist()]

    metrics = {
        "meta": {
            "generated_at": datetime.datetime.now().isoformat(timespec="seconds"),
            "subset_dir": subset_dir, "seed": seed, "resolution": res,
            "hub_indeg_cap": hub_indeg_cap, "hub_outdeg_cap": hub_outdeg_cap, "hub_weight": hub_weight,
            "engine": engine,
            "hub_filter_info": hub_filter_info, "degree_budget_info": prune_info,
            "resolutions_tried": list(resolutions),
            "total_secs": round(time.time() - t_start, 1),
        },
        "subset_meta": subset_meta,
        "sweep": sweep,
        "global": glob,
        "per_community": per_json,
        "community_labels": {str(k): v for k, v in labels.items()},
        "top_inter_pairs": top_pairs,
        "hub_rows": hub_rows,
        "hub_stats": hub_stats,
        "hierarchy": hier,
        "full_graph_stats": read_json(full_stats_json, None) if full_stats_json else None,
    }
    write_json(os.path.join(out_dir, "metrics.json"), metrics)

    from .report import make_plots, write_report
    plots = make_plots(out_dir, per, glob, hier)
    ctx = {
        "subset_name": os.path.basename(subset_dir.rstrip("/")),
        "subset_meta": subset_meta, "glob": glob, "per": per, "labels": labels,
        "top_pairs": top_pairs, "hub_rows": hub_rows, "hub_stats": hub_stats,
        "hier": hier, "sweep": sweep,
        "meta": {**metrics["meta"], "dump_date": (subset_meta.get("dump_date") or "?")},
    }
    ctx["plots"] = plots
    write_report(os.path.join(out_dir, "report.md"), ctx)
    # cleanup work file (regenerable from subset)
    try:
        del E, E_raw
        os.remove(E_raw_path)
    except Exception:
        pass
    print(f"[analyze] done -> {out_dir} ({metrics['meta']['total_secs']}s)")
    return metrics


In [ ]:
%%writefile wu/report.py
"""Plots + markdown report generation (matplotlib Agg, English labels to avoid
missing CJK fonts; narrative text is Japanese in report.md)."""
from __future__ import annotations

import os

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np


def _md_table(headers, rows):
    out = ["| " + " | ".join(str(h) for h in headers) + " |",
           "|" + "|".join("---" for _ in headers) + "|"]
    for r in rows:
        out.append("| " + " | ".join(str(x) for x in r) + " |")
    return "\n".join(out)


def make_plots(out_dir: str, per: dict, glob: dict, hier, degree_hist=None):
    os.makedirs(out_dir, exist_ok=True)
    n_c = per["N_c"]
    keep = n_c > 0

    # 1. community size histogram
    fig, ax = plt.subplots(figsize=(7, 4))
    sizes = n_c[keep]
    ax.hist(sizes, bins=np.logspace(np.log10(max(1, sizes.min())), np.log10(sizes.max()), 50))
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("community size (nodes)")
    ax.set_ylabel("#communities")
    ax.set_title(f"Community size distribution (C={glob['n_communities']})")
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(os.path.join(out_dir, "size_hist.png"), dpi=110)
    plt.close(fig)

    # 2. out_ratio distribution (unweighted + node-weighted)
    fig, ax = plt.subplots(figsize=(7, 4))
    orr = per["out_ratio_c"][keep]
    w = n_c[keep].astype(float)
    ax.hist(orr, bins=40, range=(0, 1), weights=np.ones_like(orr) / len(orr),
            alpha=0.6, label="per-community (uniform)")
    ax.hist(orr, bins=40, range=(0, 1), weights=w / w.sum(),
            alpha=0.6, label="node-weighted")
    ax.set_xlabel("out_ratio = E_out/(E_in+E_out)")
    ax.set_ylabel("share")
    ax.set_title("Inter-community out_ratio distribution")
    ax.legend()
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(os.path.join(out_dir, "out_ratio_hist.png"), dpi=110)
    plt.close(fig)

    # 3. conductance distribution
    fig, ax = plt.subplots(figsize=(7, 4))
    cc = per["conductance_c"][keep]
    ax.hist(cc[~np.isnan(cc)], bins=40)
    ax.set_xlabel("conductance = E_out/(2*E_in+E_out)")
    ax.set_ylabel("#communities")
    ax.set_title("Conductance distribution")
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(os.path.join(out_dir, "conductance_hist.png"), dpi=110)
    plt.close(fig)

    # 4. E_in vs E_out scatter (independence map)
    fig, ax = plt.subplots(figsize=(7, 5))
    m = keep & (per["E_in_c"] > 0)
    ax.scatter(per["E_in_c"][m], per["E_out_c"][m], s=np.clip(n_c[m] / 20, 3, 200),
               c=per["out_ratio_c"][m], cmap="viridis", alpha=0.7)
    lim = max(per["E_in_c"][m].max(), per["E_out_c"][m].max())
    ax.plot([1, lim], [1, lim], "k--", lw=1, label="E_out = E_in")
    ax.plot([1, lim], [1, lim * 0.2], "g:", lw=1.5, label="out_ratio = 0.17")
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("E_in (internal edges)")
    ax.set_ylabel("E_out (inter-community edges)")
    ax.set_title("Independence map (marker size ~ N_c, color = out_ratio)")
    ax.legend()
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(os.path.join(out_dir, "ein_vs_eout.png"), dpi=110)
    plt.close(fig)

    # 5. boundary/ext leakage
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(per["ext_ratio_c"][keep & ~np.isnan(per["ext_ratio_c"])], bins=40, range=(0, 1))
    ax.set_xlabel("ext_ratio = E_ext/(E_in+E_out+E_ext)  [links leaving the SUBSET]")
    ax.set_ylabel("#communities")
    ax.set_title("Subset-leakage ratio per community")
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(os.path.join(out_dir, "ext_ratio_hist.png"), dpi=110)
    plt.close(fig)

    # 6. hierarchy scatter
    if hier and hier.get("clusters"):
        rows = hier["clusters"]
        fig, ax = plt.subplots(figsize=(7, 5))
        xs = [r["N_nodes"] for r in rows]
        ys = [(r["out_ratio2"] if r["out_ratio2"] is not None else np.nan) for r in rows]
        ss = [max(20, r["N_nodes"] / max(xs) * 900) for r in rows]
        ax.scatter(xs, ys, s=ss, alpha=0.65, c=range(len(rows)), cmap="tab20")
        for r in rows[:12]:
            if r["out_ratio2"] is not None:
                ax.annotate(f"C{r['cluster']}", (r["N_nodes"], r["out_ratio2"]), fontsize=8)
        ax.set_xscale("log")
        ax.set_xlabel("cluster size (nodes)")
        ax.set_ylabel("out_ratio at level-2")
        ax.set_title(f"Level-2 clusters (K={hier['global']['n_clusters']})")
        ax.grid(alpha=0.3)
        fig.tight_layout()
        fig.savefig(os.path.join(out_dir, "hierarchy.png"), dpi=110)
        plt.close(fig)
    return [f for f in sorted(os.listdir(out_dir)) if f.endswith(".png")]


def fmt(x, nd=3):
    if x is None:
        return "-"
    if isinstance(x, float):
        if np.isnan(x):
            return "nan"
        return f"{x:.{nd}f}"
    return f"{x:,}" if isinstance(x, (int, np.integer)) else str(x)


def write_report(path: str, ctx: dict):
    """ctx keys: subset_name, subset_meta, glob, per, labels, top_pairs, hub_rows,
    hub_stats, hier, sweep, full_stats(optional), meta(extra provenance)."""
    g = ctx["glob"]
    per = ctx["per"]
    sm = ctx.get("subset_meta", {})
    lines = []
    A = lines.append
    A(f"# コミュニティ分割 PoC レポート: {ctx['subset_name']}")
    A("")
    A(f"生成: {ctx['meta'].get('generated_at','')} / seed={ctx['meta'].get('seed')} / "
      f"resolution={ctx['meta'].get('resolution')} / dump={ctx['meta'].get('dump_date','?')}")
    A("")
    A("## 1. サブセット概要")
    A("")
    A(_md_table(["項目", "値"], [
        ["戦略", sm.get("strategy", "?")],
        ["ノード数", fmt(sm.get("n_nodes"))],
        ["内部エッジ(有向・解決後)", fmt(sm.get("n_edges_internal_directed"))],
        ["内部エッジ(無向・一意)", fmt(g.get("n_edges_internal_undirected"))],
        ["サブセット外へのリンク(outgoing)", fmt(sm.get("n_edges_outgoing"))],
        ["サブセット外からのリンク(incoming)", fmt(sm.get("n_edges_incoming"))],
        ["リーク率 out/(int+out)", fmt(sm.get("leak_ratio_out"))],
        ["ページID範囲", f"{fmt(sm.get('page_id_min'))}〜{fmt(sm.get('page_id_max'))}"],
    ]))
    A("")
    A("## 2. resolution スイープ(コミュニティサイズ分布)")
    A("")
    A(_md_table(["resolution", "#comms", "最大シェア", "top5シェア", "サイズ中央値", "p25", "p75", "cross_edge_fraction"],
                [[s["resolution"], s["n_communities"], fmt(s["largest_comm_share"]),
                  fmt(s["top5_comm_share"]), fmt(s["p50"]), fmt(s["p25"]), fmt(s["p75"]),
                  fmt(s["cross_edge_fraction"])] for s in ctx.get("sweep", [])]))
    A("")
    A("## 3. 全体指標(採用 resolution)")
    A("")
    A(_md_table(["指標", "値"], [
        ["コミュニティ数", fmt(g.get("n_communities"))],
        ["最大コミュニティのノードシェア", fmt(g.get("largest_comm_share"))],
        ["上位5コミュニティのシェア", fmt(g.get("top5_comm_share"))],
        ["cross_edge_fraction(全体)", fmt(g.get("cross_edge_fraction"))],
        ["out_ratio(エッジ重み付き平均)", fmt(g.get("mean_out_ratio_edge_weighted"))],
        ["out_ratio(ノード重み付き中央値)", fmt(g.get("node_weighted_median_out_ratio"))],
        ["out_ratio p25/p50/p75", "/".join(fmt(g.get("out_ratio_percentiles", {}).get(f"p{p}")) for p in (25, 50, 75))],
        ["conductance p25/p50/p75", "/".join(fmt(g.get("conductance_percentiles", {}).get(f"p{p}")) for p in (25, 50, 75))],
        ["サブセット外へのリンク数(有向)", fmt(g.get("n_edges_to_outside_subset"))],
        ["サブセット外からのリンク数(有向)", fmt(g.get("n_edges_from_outside_subset"))],
    ]))
    A("")
    A("## 4. 上位コミュニティ(サイズ順 top 15)")
    A("")
    order = np.argsort(-per["N_c"])[:15]
    A(_md_table(["comm", "代表記事", "N_c", "E_in", "E_out", "E_ext(場外)", "out_ratio",
                 "conductance", "avg_deg", "境界ノード率"],
                [[int(c), "、".join(ctx["labels"].get(int(c), [])[:3]),
                  fmt(int(per["N_c"][c])), fmt(int(per["E_in_c"][c])), fmt(int(per["E_out_c"][c])),
                  fmt(int(per["E_ext_c"][c])), fmt(float(per["out_ratio_c"][c])),
                  fmt(float(per["conductance_c"][c])), fmt(float(per["avg_degree_c"][c]), 1),
                  fmt(float(per["boundary_node_ratio_c"][c]))] for c in order if per["N_c"][c] > 0]))
    A("")
    A("## 5. コミュニティ間エッジ top 20")
    A("")
    A(_md_table(["comm A", "comm B", "エッジ数", "A代表", "B代表"],
                [[p["comm_a"], p["comm_b"], fmt(p["edges"]),
                  "、".join(ctx["labels"].get(p["comm_a"], [])[:2]),
                  "、".join(ctx["labels"].get(p["comm_b"], [])[:2])]
                 for p in ctx.get("top_pairs", [])]))
    A("")
    A("## 6. ハブ記事の影響(次数 top 20)")
    A("")
    A(_md_table(["#", "記事", "内部次数", "場外次数", "comm", "フラグ"],
                [[h["rank"], h["title"], fmt(h["deg_internal"]), fmt(h["deg_outside"]),
                  h["comm"], ",".join(h["flags"])] for h in ctx.get("hub_rows", [])[:20]]))
    hs = ctx.get("hub_stats", {})
    A("")
    A(f"- 上位1%ノードが接する内部エッジのシェア: {fmt(hs.get('top1pct_internal_edge_incidence_share'))}")
    A(f"- 内部次数 max: {fmt(hs.get('max_internal_degree'))} / "
      f"p50・p90・p99: {fmt(hs.get('degree_percentiles', {}).get('p50'))}・"
      f"{fmt(hs.get('degree_percentiles', {}).get('p90'))}・{fmt(hs.get('degree_percentiles', {}).get('p99'))}")
    A("")
    A("## 7. 階層化テスト(level-2)")
    A("")
    hier = ctx.get("hier")
    if hier:
        hg = hier["global"]
        A(_md_table(["指標", "値"], [
            ["上位クラスタ数 K", fmt(hg.get("n_clusters"))],
            ["最大クラスタのシェア", fmt(hg.get("largest_cluster_share"))],
            ["out_ratio2(エッジ重み付き平均)", fmt(hg.get("mean_out_ratio2_edge_weighted"))],
            ["コミュニティ間グラフ modularity", fmt(hg.get("modularity_community_graph"))],
        ]))
        A("")
        rows = sorted(hier["clusters"], key=lambda r: -r["N_nodes"])[:15]
        A(_md_table(["cluster", "#comms", "N_nodes", "E_in(記事)", "E_between(comm間)",
                     "E_out2", "E_ext(場外)", "out_ratio2", "conductance2"],
                    [[r["cluster"], r["n_comms"], fmt(r["N_nodes"]), fmt(r["E_in_articles"], 0),
                      fmt(r["E_between_comms"], 0), fmt(r["E_out2"], 0), fmt(r["E_ext_outside_subset"], 0),
                      fmt(r["out_ratio2"]), fmt(r["conductance2"])] for r in rows]))
    else:
        A("(コミュニティ数が少なく階層化テストはスキップ)")
    A("")
    A("## 8. 図")
    A("")
    for p in ctx.get("plots", []):
        A(f"![{p}]({p})")
    A("")
    with open(path, "w") as f:
        f.write("\n".join(lines))
    return path


In [ ]:
%%writefile wu/cli.py
"""CLI entry point: python -m wu.cli <command> [options]

Data layout (policy 2026-09-25, see wu/paths.py):
  <base>/dump/  <base>/parsed/  <base>/graph/(+subsets/)  <base>/community/

Commands:
  download   Download dump files (page, redirect, linktarget, pagelinks[, categorylinks])
  parse      Build parsed artifacts from page/linktarget/redirect dumps
  edges      Stream pagelinks -> resolved ns0 edge list (checkpointed)
  stats      Full-graph degree/hub statistics
  subset-id  Extract page-id-window subset (contrast/baseline use)
  subset-bfs Extract BFS subset from seed titles
  analyze    Leiden + metrics + hierarchy + report for one subset
"""
from __future__ import annotations

import argparse
import datetime
import os
import sys

from .dumpio import download_all, read_json, write_json
from .paths import Dirs


def cmd_download(a):
    dirs = Dirs(a.base)
    dirs.ensure(dirs.dump)
    download_all(a.files.split(","), dirs.dump, workers=a.workers)


def cmd_parse(a):
    from .dumpio import FILES
    from .sqlparse import (build_linktarget_artifacts, build_page_artifacts,
                           build_redirect_artifacts)
    dirs = Dirs(a.base)
    dirs.ensure(dirs.parsed)
    build_page_artifacts(dirs.dump_file(FILES["page"]), dirs.parsed, dirs.meta)
    build_linktarget_artifacts(dirs.dump_file(FILES["linktarget"]), dirs.parsed, dirs.meta)
    build_redirect_artifacts(dirs.dump_file(FILES["redirect"]), dirs.parsed, dirs.meta)
    meta = read_json(dirs.meta, {}) or {}
    meta["dump_date"] = a.dump_date or meta.get("dump_date") or "unknown"
    meta["parsed_at"] = datetime.datetime.now().isoformat(timespec="seconds")
    meta.setdefault("files", {})
    for k in ("page", "redirect", "linktarget", "pagelinks"):
        f = dirs.dump_file(FILES[k])
        if os.path.exists(f):
            meta["files"][k] = {"bytes": os.path.getsize(f), "name": FILES[k]}
    meta.setdefault("layout", {"policy": "2026-09-25",
                               "dirs": {"dump": dirs.dump, "parsed": dirs.parsed,
                                        "graph": dirs.graph, "community": dirs.community}})
    write_json(dirs.meta, meta)


def cmd_edges(a):
    from .buildedges import build_edges
    from .dumpio import FILES
    dirs = Dirs(a.base)
    dirs.ensure(dirs.graph)
    build_edges(dirs.dump_file(FILES["pagelinks"]), dirs.parsed,
                dirs.edges_bin, dirs.edges_ckpt, chunk_bytes=a.chunk_bytes)


def cmd_stats(a):
    from .stats import full_stats
    full_stats(Dirs(a.base))


def cmd_subset_id(a):
    from .subsets import extract_subset, select_id_window
    dirs = Dirs(a.base)
    pids = select_id_window(dirs.parsed, a.k, a.mode)
    name = a.name or f"idwin_{a.mode}_{a.k // 1000}k"
    out = dirs.subset(name)
    dirs.ensure(out)
    meta = {"strategy": f"page_id_window({a.mode})", "k": a.k,
            "dump_date": _dump_date(dirs), "created_at": datetime.datetime.now().isoformat(timespec="seconds")}
    extract_subset(dirs.edges_bin, pids, out, meta, parsed_dir=dirs.parsed)


def cmd_subset_bfs(a):
    from .subsets import bfs_nodes, extract_subset, title_to_pid
    dirs = Dirs(a.base)
    seeds = [s.strip() for s in a.seeds.split(",") if s.strip()]
    seed_pids, missing = [], []
    for s in seeds:
        p = title_to_pid(dirs.parsed, s)
        (seed_pids.append(p) if p else missing.append(s))
    if missing:
        print(f"[warn] seeds not found: {missing}")
    if not seed_pids:
        sys.exit("no valid seeds")
    nodes, info = bfs_nodes(dirs.edges_bin, seed_pids,
                            max_nodes=a.max_nodes, cap_per_node=a.cap,
                            max_hops=a.hops, rng_seed=a.seed,
                            parsed_dir=dirs.parsed, graph_dir=dirs.graph,
                            max_in_degree=a.max_in_degree)
    name = a.name or f"bfs_{len(nodes) // 1000}k"
    out = dirs.subset(name)
    dirs.ensure(out)
    meta = {"strategy": "outlink_bfs", "seeds": seeds, "seeds_missing": missing,
            "seed_page_ids": seed_pids, "max_nodes": a.max_nodes, "cap_per_node": a.cap,
            "max_hops": a.hops, "rng_seed": a.seed, "bfs_info": info,
            "dump_date": _dump_date(dirs), "created_at": datetime.datetime.now().isoformat(timespec="seconds")}
    extract_subset(dirs.edges_bin, nodes, out, meta, parsed_dir=dirs.parsed)


def cmd_analyze(a):
    from .pipeline import analyze_subset
    dirs = Dirs(a.base)
    subset_dir = a.subset_dir or dirs.subset(a.subset)
    out_dir = a.out_dir or dirs.community_run(os.path.basename(subset_dir.rstrip("/")))
    dirs.ensure(out_dir)
    resolutions = tuple(float(x) for x in a.resolutions.split(","))
    analyze_subset(subset_dir, out_dir, resolutions=resolutions, seed=a.seed,
                   full_stats_json=None, dirs=dirs,
                   hub_indeg_cap=a.hub_indeg_cap, hub_outdeg_cap=a.hub_outdeg_cap,
                   degree_budget=a.degree_budget, degree_budget_mode=a.degree_budget_mode,
                   hub_weight=a.hub_weight, engine=a.engine)


def _dump_date(dirs: Dirs) -> str:
    m = read_json(dirs.meta, {}) or {}
    return m.get("dump_date", "?")


def main():
    ap = argparse.ArgumentParser(prog="wu")
    ap.add_argument("--base", default="data",
                    help="base dir; layout: dump/ parsed/ graph/ community/ (see wu/paths.py)")
    sub = ap.add_subparsers(dest="cmd", required=True)

    p = sub.add_parser("download"); p.add_argument("--files", default="page,redirect,linktarget,pagelinks")
    p.add_argument("--workers", type=int, default=3); p.set_defaults(fn=cmd_download)

    p = sub.add_parser("parse"); p.add_argument("--dump-date", default=None); p.set_defaults(fn=cmd_parse)
    p = sub.add_parser("edges"); p.add_argument("--chunk-bytes", type=int, default=1 << 24); p.set_defaults(fn=cmd_edges)
    p = sub.add_parser("stats"); p.set_defaults(fn=cmd_stats)

    p = sub.add_parser("subset-id")
    p.add_argument("--k", type=int, default=10000); p.add_argument("--mode", default="late",
                    choices=["late", "early", "middle"]); p.add_argument("--name", default=None)
    p.set_defaults(fn=cmd_subset_id)

    p = sub.add_parser("subset-bfs")
    p.add_argument("--seeds", required=True); p.add_argument("--max-nodes", type=int, default=30000)
    p.add_argument("--cap", type=int, default=40); p.add_argument("--hops", type=int, default=4)
    p.add_argument("--max-in-degree", type=int, default=10000)
    p.add_argument("--seed", type=int, default=42); p.add_argument("--name", default=None)
    p.set_defaults(fn=cmd_subset_bfs)

    p = sub.add_parser("analyze")
    p.add_argument("--subset", default=None, help="subset name under <base>/graph/subsets/")
    p.add_argument("--subset-dir", default=None, help="explicit subset dir (overrides --subset)")
    p.add_argument("--out-dir", default=None, help="default: <base>/community/<subset>")
    p.add_argument("--resolutions", default="0.5,1.0,2.0")
    p.add_argument("--seed", type=int, default=42)
    p.add_argument("--hub-indeg-cap", type=int, default=None)
    p.add_argument("--hub-outdeg-cap", type=int, default=None)
    p.add_argument("--degree-budget", type=int, default=None)
    p.add_argument("--degree-budget-mode", default="smart", choices=["smart", "both"])
    p.add_argument("--hub-weight", default=None, help="'log' or 'deg<alpha>' e.g. deg1, deg0.5")
    p.add_argument("--engine", default="igraph", choices=["igraph", "leidenalg"])
    p.set_defaults(fn=cmd_analyze)

    a = ap.parse_args()
    a.fn(a)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile wu/__init__.py
"""wikiuniverse (wu): PoC toolkit for jawiki article-graph community analysis.

Design constraints:
- Low RAM (streaming parsers, memmap, sorted-array joins via searchsorted).
- Deterministic (fixed seeds, blake2b64 title hashes, recorded metadata).
- Artifacts on disk: parquet / npy / raw int32 binary edge lists.
"""
__version__ = "0.1.0"


In [ ]:
# 3. データ取得(page / redirect / linktarget / pagelinks, 計 ~1.13GB, resume 対応)
from wu.dumpio import download_all
download_all(["page", "redirect", "linktarget", "pagelinks"], "data/dump", workers=4)
!ls -la data/dump/

In [ ]:
# 4. ページ正規化: page/linktarget/redirect を解析 → ハッシュ表・リダイレクト連鎖解決
from wu.cli import main as wu_main
import sys
def wu(*args):
    sys.argv = ["wu", "--base", "data", *args]
    wu_main()

wu("parse", "--dump-date", "2026-09-02")  # ← ダンプ日付は latest の実値に更新すること
import json; print(json.dumps(json.load(open("data/parsed/meta.json")), ensure_ascii=False, indent=1))

In [ ]:
# 5. リンクグラフ構築: pagelinks → 解決済み ns0→ns0 エッジ(赤リンク/リダイレクト/セルフループ処理)
wu("edges")
import os; print("edges_ns0.bin:", os.path.getsize("data/graph/edges_ns0.bin")/1e9, "GB")

In [ ]:
# 6. 基本統計(全グラフ次数分布・ハブ top)
wu("stats")
import json
d = json.load(open("data/graph/full_stats.json"))
print(json.dumps(d["stats"], ensure_ascii=False, indent=1))
for h in d["top_hubs"][:25]:
    print(f'{h["rank"]:3d}. {h["title"][:40]:42s} in={h["in"]:7,d} out={h["out"]:6,d} {",".join(h["flags"])}')

In [ ]:
# 7a. スモークテスト用サブセット: 地理系 BFS 30k(1-2分)
wu("subset-bfs", "--seeds", "東京都,大阪府,京都府,北海道,福岡県,愛知県,宮城県,広島県,新潟県,長野県,日本の地理,市町村",
   "--max-nodes", "30000", "--cap", "40", "--hops", "3", "--max-in-degree", "10000", "--name", "bfs_geo_30k")

In [ ]:
# 7b. 対照実験: ID窓サブセット(リーク率が跳ね上がることを確認 — 構造評価には使用禁止)
wu("subset-id", "--k", "10000", "--mode", "late", "--name", "idwin_late_10k")

In [ ]:
# 8-11. Leiden(resolution スイープ)→ 指標 → ハブ → L2 階層化 → レポート/プロット
wu("analyze", "--subset", "bfs_geo_30k", "--out-dir", "results/bfs_geo_30k",
   "--resolutions", "0.5,1.0,2.0", "--engine", "igraph")

In [ ]:
# 9. ヒストグラム表示
from IPython.display import display, Image, Markdown
import glob
for p in sorted(glob.glob("results/bfs_geo_30k/*.png")):
    display(Image(p))

In [ ]:
# 10-11. 数値確認: 全体指標 / top ペア / 階層
import json
m = json.load(open("results/bfs_geo_30k/metrics.json"))
print("== global ==");  print(json.dumps(m["global"], ensure_ascii=False, indent=1))
print("== sweep ==");    print(json.dumps(m["sweep"], ensure_ascii=False, indent=1))
print("== top pairs =="); 
lab = m["community_labels"]
for p in m["top_inter_pairs"][:15]:
    print(p["edges"], "|", ",".join(lab.get(str(p["comm_a"]),[])[:2]), "<->", ",".join(lab.get(str(p["comm_b"]),[])[:2]))
print("== hierarchy =="); print(json.dumps(m["hierarchy"]["global"] if m["hierarchy"] else None, ensure_ascii=False, indent=1))
display(Markdown(open("results/bfs_geo_30k/report.md").read()))

In [ ]:
# 12. 本番 PoC: 100k ノード(Colab RAM 注意: igraph エンジンのとき ~600MB)
wu("subset-bfs", "--seeds", "東京都,大阪府,京都府,北海道,福岡県,愛知県,宮城県,広島県,新潟県,長野県,日本の地理,市町村",
   "--max-nodes", "100000", "--cap", "60", "--hops", "4", "--max-in-degree", "10000", "--name", "bfs_geo_100k")
wu("analyze", "--subset", "bfs_geo_100k", "--out-dir", "results/bfs_geo_100k_raw",
   "--resolutions", "0.5,1.0,2.0", "--engine", "igraph")

In [ ]:
# 12b. レイアウト用グラフ実験: smart 次数予算剪定(銀河を壊さずハブ間高速道路のみ剪定)
wu("analyze", "--subset", "bfs_geo_100k", "--out-dir", "results/bfs_geo_100k_smart40",
   "--degree-budget", "40", "--degree-budget-mode", "smart", "--resolutions", "1.0", "--engine", "igraph")

In [ ]:
# 13. 結果の保存(Google Drive へ)
# from google.colab import drive; drive.mount('/content/drive')
# !cp -r results /content/drive/MyDrive/wikiuniverse_results
!zip -qr results.zip results data/parsed/meta.json data/graph/full_stats.json
print("done")